# LeetCode 3916 — Number of ZigZag Arrays III bằng Python

- Link: https://leetcode.com/problems/number-of-zigzag-arrays-iii/
- Độ khó: **Hard** · 🔒 **Premium**
- Chủ đề: Quy hoạch động, tổng tiền tố, đa thức nội suy Lagrange, tổ hợp (cơ sở nhị thức / sai phân Newton), số học modulo

> **Ghi chú Premium:** đề bài trong notebook được viết lại theo hiểu biết về bài (dựa trên bản mô tả công khai của doocs). Nếu có tài khoản Premium, nên đối chiếu lại đề gốc. Skeleton dưới đây **suy luận** theo hai bài cùng loạt (*Number of ZigZag Arrays I/II*), cần đối chiếu tên phương thức.

Skeleton (suy luận):

```python
class Solution:
    def zigZagArrays(self, n: int, l: int, r: int) -> int:
```

## 1. Mục tiêu học

1. Nhận ra rằng điều kiện ZigZag chỉ phụ thuộc **thứ tự tương đối** của các phần tử, nên đáp án chỉ phụ thuộc `n` và `m = r - l + 1`.
2. Viết DP đếm dãy zigzag với **tổng tiền tố** `O(n · m)` (giải được khi `m` nhỏ).
3. Chứng minh `f(m)` (số dãy zigzag độ dài `n` với giá trị trong `[1..m]`) là **đa thức bậc ≤ n theo `m`**.
4. Dùng **nội suy Lagrange** tính `f(m)` với `m` tới `10^9` chỉ từ `n + 1` giá trị nhỏ.
5. Hiểu cách viết khác bằng **cơ sở nhị thức** `f(m) = Σ_j C(m, j) · g(j)` (sai phân Newton) và nghịch đảo modulo.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho ba số nguyên `n`, `l`, `r`. Một mảng độ dài `n` là **ZigZag** nếu:

- mọi phần tử nằm trong đoạn `[l, r]`;
- không có hai phần tử kề nhau bằng nhau;
- không có ba phần tử liên tiếp **tăng ngặt** hoặc **giảm ngặt**.

Đếm số mảng ZigZag, trả về theo modulo `10^9 + 7`.

Hai điều kiện cuối nghĩa là chiều đi **luân phiên**: lên, xuống, lên, xuống… hoặc xuống, lên, xuống, lên…

**Ví dụ 1:** `n = 3, l = 4, r = 5` → `2` (`[4,5,4]`, `[5,4,5]`).

**Ví dụ 2:** `n = 3, l = 1, r = 3` → `10`.

**Ràng buộc:** `3 ≤ n ≤ 200`, `1 ≤ l < r ≤ 10^9`.

In [1]:
from itertools import product, accumulate
import random, time

MOD = 10**9 + 7
examples = [(3, 4, 5, 2), (3, 1, 3, 10)]
for n, l, r, exp in examples:
    print(f"n={n}, l={l}, r={r}, m=r-l+1={r-l+1} -> kỳ vọng {exp}")

n=3, l=4, r=5, m=r-l+1=2 -> kỳ vọng 2
n=3, l=1, r=3, m=r-l+1=3 -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1. Kiểm tra một mảng có phải ZigZag

Dịch đúng định nghĩa: kề nhau khác nhau, và mỗi bộ ba `(a, b, c)` không đơn điệu ngặt. Hàm này là "trọng tài" cho mọi cách giải.

In [2]:
def is_zigzag(a):
    for i in range(len(a) - 1):
        if a[i] == a[i + 1]:
            return False
    for i in range(len(a) - 2):
        x, y, z = a[i], a[i + 1], a[i + 2]
        if x < y < z or x > y > z:
            return False
    return True

print(is_zigzag([1, 3, 2]), is_zigzag([1, 2, 3]), is_zigzag([2, 2, 1]), is_zigzag([2, 1, 3, 1]))

True False False True


### 3.2. Tổng tiền tố với `itertools.accumulate`

`accumulate([a0, a1, a2])` cho `[a0, a0+a1, a0+a1+a2]`. Thêm `initial=0` để có `pre[i] = tổng i phần tử đầu`. Khi đó tổng đoạn `a[i..j-1] = pre[j] - pre[i]`, tính trong `O(1)`.

In [3]:
a = [3, 1, 4, 1, 5]
pre = list(accumulate(a, initial=0))
print(pre, "| tổng a[1..3] =", pre[4] - pre[1])

[0, 3, 4, 8, 9, 14] | tổng a[1..3] = 6


### 3.3. Nghịch đảo modulo bằng `pow(x, MOD - 2, MOD)`

`MOD = 10^9 + 7` là số nguyên tố, nên theo định lý Fermat nhỏ: `x^{-1} ≡ x^{MOD-2} (mod MOD)` khi `x` không chia hết cho `MOD`. Python có sẵn `pow(x, -1, MOD)` (từ 3.8) cũng cho cùng kết quả.

In [4]:
x = 12345
inv = pow(x, MOD - 2, MOD)
print(inv, inv == pow(x, -1, MOD), x * inv % MOD)

203483193 True 1


### 3.4. Nội suy Lagrange

Một đa thức bậc `≤ d` được xác định duy nhất bởi `d + 1` điểm. Với các điểm `x = 0, 1, …, d` và giá trị `y_i`, giá trị tại `X` là

`P(X) = Σ_i y_i · Π_{j≠i} (X - j) / (i - j)`.

Mẫu số `Π_{j≠i}(i - j) = i! · (d - i)! · (-1)^{d-i}`; tử số dùng **tích tiền tố/hậu tố** của `(X - j)` để tính trong `O(d)` tổng cộng.

In [5]:
def lagrange_consecutive(ys, X, mod=MOD):
    """ys[i] = P(i) với i = 0..d. Trả về P(X) mod mod trong O(d)."""
    d = len(ys) - 1
    if 0 <= X <= d:
        return ys[X] % mod
    pre = [1] * (d + 2)
    suf = [1] * (d + 2)
    for i in range(d + 1):
        pre[i + 1] = pre[i] * ((X - i) % mod) % mod
    for i in range(d, -1, -1):
        suf[i] = suf[i + 1] * ((X - i) % mod) % mod
    fact = [1] * (d + 1)
    for i in range(1, d + 1):
        fact[i] = fact[i - 1] * i % mod
    res = 0
    for i in range(d + 1):
        num = pre[i] * suf[i + 1] % mod
        den = fact[i] * fact[d - i] % mod
        term = ys[i] * num % mod * pow(den, mod - 2, mod) % mod
        res = (res - term) % mod if (d - i) & 1 else (res + term) % mod
    return res

# kiểm tra với P(x) = x^3 - 2x + 7
P = lambda x: x**3 - 2 * x + 7
ys = [P(i) for i in range(4)]
print(lagrange_consecutive(ys, 100), P(100) % MOD, lagrange_consecutive(ys, 10**9) == P(10**9) % MOD)

999807 999807 True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

**Quan sát 1 — dịch chuyển đoạn.** Điều kiện ZigZag chỉ so sánh `<`, `>`, `=`. Trừ mọi phần tử cho `l - 1` không làm đổi các so sánh, nên đáp án chỉ phụ thuộc `n` và `m = r - l + 1` (đưa về đoạn `[1..m]`). Gọi nó là `f_n(m)`.

**Tính tay ví dụ 2** (`n = 3`, `m = 3`): dãy dạng **lên–xuống** `a < b > c`:

- `b = 2`: `a = 1`, `c = 1` → 1 dãy.
- `b = 3`: `a ∈ {1,2}`, `c ∈ {1,2}` → 4 dãy.

Tổng lên–xuống = 5. Do đối xứng `v ↦ m + 1 - v`, xuống–lên cũng 5. Tổng `10` ✔.

**Tổng quát cho `n = 3`:** lên–xuống = `Σ_{b=1}^{m} (b-1)^2 = (m-1)m(2m-1)/6` — một **đa thức bậc 3** theo `m`. Đây là gợi ý lớn: liệu `f_n(m)` có luôn là đa thức theo `m` không?

In [6]:
def f3_formula(m):
    return 2 * (m - 1) * m * (2 * m - 1) // 6

print([f3_formula(m) for m in range(1, 7)])

[0, 2, 10, 28, 60, 110]


## 5. Cách 1 — Brute Force: liệt kê mọi mảng

Duyệt toàn bộ `m^n` mảng bằng `itertools.product` và đếm mảng hợp lệ. Chỉ chạy được khi `m^n` nhỏ (vài trăm nghìn).

In [7]:
def brute(n, l, r):
    m = r - l + 1
    return sum(1 for a in product(range(m), repeat=n) if is_zigzag(a)) % MOD

for n, l, r, exp in examples:
    print(brute(n, l, r), "kỳ vọng", exp)
print("so với công thức n=3:", [brute(3, 1, m) for m in range(1, 7)])

2 kỳ vọng 2
10 kỳ vọng 10
so với công thức n=3: [0, 2, 10, 28, 60, 110]


**Độ phức tạp:** `O(m^n · n)`. Với `m = 10^9` hay `n = 200` là bất khả thi.

**Nút thắt:** (1) liệt kê từng mảng; (2) giá trị `m` quá lớn — ngay cả DP theo giá trị cũng tốn `O(n · m)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — DP theo (vị trí, giá trị cuối, chiều cuối).** Gọi `up[v]` = số dãy (độ dài hiện tại) kết thúc tại giá trị `v` với bước cuối **đi lên**, `down[v]` tương tự với bước cuối **đi xuống**. Vì chiều phải luân phiên:

- `up_new[v] = Σ_{u < v} down[u]` (bước trước đi xuống, giờ đi lên tới `v`);
- `down_new[v] = Σ_{u > v} up[u]`.

Mỗi tầng tính bằng tổng tiền tố trong `O(m)` ⇒ tổng `O(n · m)`.

**Bước 2 — đột phá: `f_n(m)` là đa thức bậc ≤ n.** Mỗi mảng ZigZag dùng một tập `j` giá trị phân biệt (`j ≤ n`). "Nén" các giá trị về `1..j` (giữ thứ tự) cho một mảng ZigZag **dùng đủ** `1..j`. Ngược lại, mỗi mảng dùng đủ `1..j` cùng một tập con `j` phần tử của `[1..m]` cho đúng một mảng gốc. Vậy

`f_n(m) = Σ_{j=0}^{n} C(m, j) · g_n(j)`,

trong đó `g_n(j)` = số mảng ZigZag dùng **đủ** `1..j` (không phụ thuộc `m`). Mỗi `C(m, j)` là đa thức bậc `j` theo `m` ⇒ `f_n(m)` là đa thức bậc ≤ `n`.

**Hệ quả:** chỉ cần tính `f_n(0), f_n(1), …, f_n(n)` bằng DP (với `m ≤ n`, rất rẻ), rồi **nội suy** ra `f_n(M)` với `M = r - l + 1` lớn tùy ý.

## 7. Các cách tiếp theo

- **Cách 2:** DP + tổng tiền tố `O(n · m)` — đúng nhưng chỉ dùng được khi `m` nhỏ.
- **Cách 3:** DP cho `m = 0..n` + **nội suy Lagrange** tại `M` — `O(n³)` hoặc `O(n²·…)`, lời giải nộp chính.
- **Cách 4:** **Cơ sở nhị thức**: tính `g_n(j)` bằng bao hàm–loại trừ rồi cộng `C(M, j) · g_n(j)`.

### 7.1. Cách 2 — DP với tổng tiền tố

**Giải thích từng đoạn:**

1. Độ dài 1: chưa có "chiều". Ta khởi tạo trực tiếp độ dài 2: `up[v] = v - 1` (số `u < v`), `down[v] = m - v` (số `u > v`), với `v = 1..m` (lưu chỉ số `0..m-1`).
2. Mỗi bước tăng độ dài: tính `pre_down` (tiền tố của `down`) và `suf_up` (hậu tố của `up`), rồi `up_new[v] = pre_down[v]`, `down_new[v] = suf_up[v+1]`.
3. Đáp án: `Σ up + Σ down` ở độ dài `n`.

In [8]:
def dp_count(n, m, mod=MOD):
    """Số mảng ZigZag độ dài n (n >= 2) với giá trị trong [1..m]."""
    if m <= 0:
        return 0
    up = [v for v in range(m)]            # chỉ số v ứng với giá trị v+1: có v giá trị nhỏ hơn
    down = [m - 1 - v for v in range(m)]
    for _ in range(n - 2):
        pre_down = list(accumulate(down, initial=0))       # pre_down[v] = Σ_{u<v} down[u]
        total_up = sum(up)
        suf = list(accumulate(up, initial=0))               # Σ_{u<=v} up = suf[v+1]
        new_up = [pre_down[v] % mod for v in range(m)]
        new_down = [(total_up - suf[v + 1]) % mod for v in range(m)]   # Σ_{u>v} up[u]
        up, down = new_up, new_down
    return (sum(up) + sum(down)) % mod

for n, l, r, exp in examples:
    print(dp_count(n, r - l + 1), "kỳ vọng", exp)
print("đối chiếu brute:", all(dp_count(n, m) == brute(n, 1, m) for n in range(3, 7) for m in range(1, 6)))

2 kỳ vọng 2
10 kỳ vọng 10
đối chiếu brute: True


In [9]:
class SolutionDP:
    def zigZagArrays(self, n: int, l: int, r: int) -> int:
        m = r - l + 1           # chỉ dùng được khi m nhỏ (vài nghìn)
        return dp_count(n, m)

print([SolutionDP().zigZagArrays(n, l, r) for n, l, r, _ in examples])

[2, 10]


**Độ phức tạp Cách 2:** `O(n · m)` thời gian, `O(m)` bộ nhớ. Với `m = 10^9` là quá lớn — cần Cách 3.

### 7.2. Cách 3 — Nội suy Lagrange (lời giải chính)

**Giải thích từng đoạn:**

1. `M = r - l + 1`. Nếu `M ≤ n` thì tính thẳng `dp_count(n, M)`.
2. Ngược lại tính `ys[x] = dp_count(n, x)` cho `x = 0..n` (`n + 1` điểm, mỗi điểm `O(n · x)` ⇒ tổng `O(n³)` ≈ `8·10^6` phép tính nhỏ cho `n = 200`; nhờ `accumulate` chạy ở tốc độ C).
3. `f(M) = lagrange_consecutive(ys, M)`.

Tối ưu thêm: DP cho mọi `m` có thể dùng chung bằng đối xứng, nhưng không cần thiết với `n ≤ 200`.

In [10]:
def lagrange_solution(n, l, r):
    M = r - l + 1
    if M <= n:
        return dp_count(n, M)
    ys = [dp_count(n, x) for x in range(n + 1)]
    return lagrange_consecutive(ys, M)

for n, l, r, exp in examples:
    print(lagrange_solution(n, l, r), "kỳ vọng", exp)
print("n=3 với m lớn so với công thức:", lagrange_solution(3, 1, 10**6) == f3_formula(10**6) % MOD)

2 kỳ vọng 2
10 kỳ vọng 10
n=3 với m lớn so với công thức: True


**Độ phức tạp Cách 3:** `O(n³)` cho phần DP (thực tế nhanh nhờ `accumulate`) + `O(n log MOD)` cho nội suy; bộ nhớ `O(n)`.

### 7.3. Cách 4 — Cơ sở nhị thức (bao hàm–loại trừ / sai phân Newton)

Từ `f(m) = Σ_j C(m, j) · g(j)`, áp dụng **nghịch đảo nhị thức**:

`g(j) = Σ_{i=0}^{j} (-1)^{j-i} · C(j, i) · f(i)`.

(Ý nghĩa: `g(j)` = số mảng dùng **đủ** `j` giá trị = bao hàm–loại trừ trên các giá trị bị thiếu.) Sau đó

`f(M) = Σ_{j=0}^{n} C(M, j) · g(j)`, với `C(M, j) = M(M-1)…(M-j+1) / j!` tính dần theo `j` (`M` lớn nhưng `j ≤ n`).

Đây chính là **công thức nội suy Newton** với sai phân tiến `Δ^j f(0) = g(j)`. Bonus: `g(j)` có ý nghĩa tổ hợp rõ ràng, và `g(j) = 0` với `j > n`.

In [11]:
def binomial_basis_solution(n, l, r, mod=MOD):
    M = r - l + 1
    f = [dp_count(n, x) for x in range(n + 1)]
    # C(j, i) theo tam giác Pascal
    C = [[0] * (n + 1) for _ in range(n + 1)]
    for a in range(n + 1):
        C[a][0] = 1
        for b in range(1, a + 1):
            C[a][b] = (C[a - 1][b - 1] + C[a - 1][b]) % mod
    g = []
    for j in range(n + 1):
        s = 0
        for i in range(j + 1):
            term = C[j][i] * f[i]
            s = s - term if (j - i) & 1 else s + term
        g.append(s % mod)
    res, comb = 0, 1                     # comb = C(M, j)
    for j in range(n + 1):
        if j > 0:
            comb = comb * ((M - j + 1) % mod) % mod * pow(j, mod - 2, mod) % mod
        res = (res + comb * g[j]) % mod
    return res

for n, l, r, exp in examples:
    print(binomial_basis_solution(n, l, r), "kỳ vọng", exp)

2 kỳ vọng 2
10 kỳ vọng 10


In [12]:
# g(j) cho n = 4: số mảng ZigZag độ dài 4 dùng đủ các giá trị 1..j
n = 4
f = [dp_count(n, x) for x in range(n + 1)]
g = [sum((-1) ** (j - i) * __import__('math').comb(j, i) * f[i] for i in range(j + 1)) for j in range(n + 1)]
print("f =", f)
print("g =", g)
# kiểm chứng g bằng vét cạn
def surj(n, j):
    return sum(1 for a in product(range(j), repeat=n) if is_zigzag(a) and len(set(a)) == j)
print("vét cạn g =", [surj(n, j) for j in range(n + 1)])

f = [0, 0, 2, 16, 62]
g = [0, 0, 2, 10, 10]
vét cạn g = [0, 0, 2, 10, 10]


In [13]:
class SolutionBinomial:
    def zigZagArrays(self, n: int, l: int, r: int) -> int:
        return binomial_basis_solution(n, l, r)

print([SolutionBinomial().zigZagArrays(n, l, r) for n, l, r, _ in examples])

[2, 10]


**Độ phức tạp Cách 4:** `O(n³)` (DP) + `O(n²)` (bao hàm–loại trừ) + `O(n log MOD)`; bộ nhớ `O(n²)` cho bảng Pascal.

## 8. TRACE từng bước

Trace DP (Cách 2) với `n = 4`, `m = 3`. Mỗi tầng in `up`, `down` (chỉ số `0,1,2` ứng giá trị `1,2,3`).

| độ dài | up (giá trị 1,2,3) | down (giá trị 1,2,3) | tổng |
|---|---|---|---|
| 2 | [0, 1, 2] | [2, 1, 0] | 6 |
| 3 | [0, 2, 3] | [3, 2, 0] | 10 |
| 4 | [0, 3, 5] | [5, 3, 0] | 16 |

Ví dụ `up` ở độ dài 3, giá trị 3: `Σ_{u<3} down[u] = 2 + 1 = 3` (các dãy `x > 1 < 3` và `x > 2 < 3`…).

In [14]:
def trace_dp(n, m):
    up = [v for v in range(m)]
    down = [m - 1 - v for v in range(m)]
    print(f"độ dài 2: up={up} down={down} tổng={sum(up) + sum(down)}")
    for length in range(3, n + 1):
        pre_down = list(accumulate(down, initial=0))
        suf = list(accumulate(up, initial=0))
        total_up = suf[-1]
        up, down = [pre_down[v] for v in range(m)], [total_up - suf[v + 1] for v in range(m)]
        print(f"độ dài {length}: up={up} down={down} tổng={sum(up) + sum(down)}")

trace_dp(4, 3)
print("vét cạn n=4, m=3:", brute(4, 1, 3))

độ dài 2: up=[0, 1, 2] down=[2, 1, 0] tổng=6
độ dài 3: up=[0, 2, 3] down=[3, 2, 0] tổng=10
độ dài 4: up=[0, 3, 5] down=[5, 3, 0] tổng=16
vét cạn n=4, m=3: 16


Trace nội suy: với `n = 3`, ta có `ys = f(0..3)`, nội suy tại `M = 10` và so với DP trực tiếp.

In [15]:
n, M = 3, 10
ys = [dp_count(n, x) for x in range(n + 1)]
print("ys =", ys)
print("nội suy f(10) =", lagrange_consecutive(ys, M), "| DP trực tiếp f(10) =", dp_count(n, M))
n, M = 6, 25
ys = [dp_count(n, x) for x in range(n + 1)]
print("n=6: ys =", ys, "| nội suy f(25) =", lagrange_consecutive(ys, M), "| DP =", dp_count(n, M))

ys = [0, 0, 2, 10]
nội suy f(10) = 570 | DP trực tiếp f(10) = 570
n=6: ys = [0, 0, 2, 42, 314, 1414, 4706] | nội suy f(25) = 36652020 | DP = 36652020


## 9. Code nộp LeetCode

In [16]:
from itertools import accumulate

class Solution:
    def zigZagArrays(self, n: int, l: int, r: int) -> int:
        MOD = 10**9 + 7

        def count(m):
            # số mảng ZigZag độ dài n, giá trị trong [1..m]
            if m <= 1:
                return 0
            up = list(range(m))
            down = list(range(m - 1, -1, -1))
            for _ in range(n - 2):
                pre_down = list(accumulate(down, initial=0))
                suf = list(accumulate(up, initial=0))
                tot = suf[-1]
                up = [pre_down[v] % MOD for v in range(m)]
                down = [(tot - suf[v + 1]) % MOD for v in range(m)]
            return (sum(up) + sum(down)) % MOD

        M = r - l + 1
        if M <= n:
            return count(M)
        ys = [count(x) for x in range(n + 1)]       # f(0..n), đa thức bậc <= n
        d = n
        pre = [1] * (d + 2)
        suf = [1] * (d + 2)
        for i in range(d + 1):
            pre[i + 1] = pre[i] * ((M - i) % MOD) % MOD
        for i in range(d, -1, -1):
            suf[i] = suf[i + 1] * ((M - i) % MOD) % MOD
        fact = [1] * (d + 1)
        for i in range(1, d + 1):
            fact[i] = fact[i - 1] * i % MOD
        res = 0
        for i in range(d + 1):
            term = ys[i] * pre[i] % MOD * suf[i + 1] % MOD
            term = term * pow(fact[i] * fact[d - i] % MOD, MOD - 2, MOD) % MOD
            res = (res - term) % MOD if (d - i) & 1 else (res + term) % MOD
        return res

sol = Solution()
for n, l, r, exp in examples:
    got = sol.zigZagArrays(n, l, r)
    print((n, l, r), "->", got, "✔" if got == exp else "✘")

(3, 4, 5) -> 2 ✔
(3, 1, 3) -> 10 ✔


## 10. Vì sao thuật toán đúng

1. **Chuẩn hóa:** phép tịnh tiến `x ↦ x - l + 1` là song ánh giữa mảng trên `[l, r]` và trên `[1, m]`, giữ nguyên mọi so sánh ⇒ số mảng ZigZag bằng nhau.
2. **DP đúng:** mảng ZigZag ⇔ dãy chiều (lên/xuống) giữa các phần tử kề nhau luân phiên và không có bước "bằng". Trạng thái `(giá trị cuối, chiều cuối)` đủ để quyết định bước tiếp theo hợp lệ hay không; các chuyển trạng thái liệt kê đúng mọi giá trị kế tiếp.
3. **Đa thức:** ánh xạ "nén giá trị" là song ánh giữa {mảng ZigZag trên `[1..m]` dùng đúng tập giá trị `S`, `|S| = j`} và {mảng ZigZag dùng đủ `1..j`}, vì ZigZag chỉ phụ thuộc thứ tự. Suy ra `f(m) = Σ_{j ≤ n} C(m, j) g(j)`, đa thức bậc ≤ `n` (đẳng thức đúng với mọi số nguyên `m ≥ 0`, nên là đẳng thức đa thức).
4. **Nội suy:** đa thức bậc ≤ `n` xác định duy nhất bởi `n + 1` giá trị tại `0..n`; công thức Lagrange cho đúng giá trị tại mọi `M`, và mọi phép toán trong trường `Z_p` vẫn đúng vì mẫu số `i!(n-i)!` khác 0 mod `p` (`n < p`).

## 11. Phân tích Time / Space Complexity

Đặt `n` = độ dài mảng, `m = r - l + 1`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(m^n · n)` | `O(n)` |
| 2. DP + tiền tố | `O(n · m)` | `O(m)` |
| 3. DP nhỏ + Lagrange | `Σ_{x ≤ n} O(n · x) = O(n³)` + `O(n log p)` | `O(n)` |
| 4. Cơ sở nhị thức | `O(n³ + n²)` | `O(n²)` |

Với `n = 200`: khoảng `200 · 200 · 200 / 2 = 4·10^6` phần tử được xử lý trong `accumulate`/list comprehension — dưới 1–2 giây trong Python.

## 12. Bộ test

In [17]:
def run_all(n, l, r):
    m = r - l + 1
    res = [lagrange_solution(n, l, r), binomial_basis_solution(n, l, r), Solution().zigZagArrays(n, l, r)]
    if m <= 3000:
        res.append(dp_count(n, m))
    if m ** n <= 200000:
        res.append(brute(n, l, r))
    return res

# 12.1 ví dụ đề
for n, l, r, exp in examples:
    assert all(v == exp for v in run_all(n, l, r)), run_all(n, l, r)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [18]:
# 12.2 ca biên (giá trị kỳ vọng tự tính bằng vét cạn / công thức)
assert Solution().zigZagArrays(3, 1, 2) == 2            # m = 2 nhỏ nhất
assert Solution().zigZagArrays(4, 1, 2) == 2            # m = 2: chỉ có 1212 và 2121
assert Solution().zigZagArrays(200, 7, 8) == 2          # m = 2 với n lớn
assert Solution().zigZagArrays(3, 10**9 - 1, 10**9) == 2
assert Solution().zigZagArrays(3, 1, 10**9) == f3_formula(10**9) % MOD
assert Solution().zigZagArrays(5, 3, 7) == brute(5, 1, 5)   # tịnh tiến đoạn
print("Ca biên: OK")

Ca biên: OK


In [19]:
# 12.3 phản ví dụ cho ý tưởng sai "đáp án = 2 * C(m, n)" (chỉ đếm mảng có giá trị phân biệt kiểu tăng dần)
from math import comb
for n, m in [(3, 3), (4, 3), (4, 5)]:
    print(f"n={n}, m={m}: 2*C(m,n) = {2 * comb(m, n)}, đúng = {dp_count(n, m)}")

n=3, m=3: 2*C(m,n) = 2, đúng = 10
n=4, m=3: 2*C(m,n) = 0, đúng = 16
n=4, m=5: 2*C(m,n) = 10, đúng = 170


In [20]:
# 12.4 đối chiếu ngẫu nhiên: Lagrange/nhị thức vs DP trực tiếp (m vừa phải), vs vét cạn (m, n nhỏ)
rng = random.Random(3916)
for _ in range(60):
    n = rng.randint(3, 12)
    l = rng.randint(1, 50)
    r = l + rng.randint(1, 60)
    vals = run_all(n, l, r)
    assert len(set(vals)) == 1, (n, l, r, vals)
print("60 test ngẫu nhiên: OK")

60 test ngẫu nhiên: OK


In [21]:
# 12.5 test lớn đúng ràng buộc: n = 200, m ~ 10^9; kiểm tra Lagrange == cơ sở nhị thức và thời gian
t0 = time.time(); a = Solution().zigZagArrays(200, 1, 10**9); t1 = time.time()
b = binomial_basis_solution(200, 1, 10**9)
print("đáp án:", a, "| khớp cách 4:", a == b, "| thời gian < 5s:", t1 - t0 < 5)
# Kiểm tra tính đa thức: nội suy từ 0..n rồi so với DP tại m = n + 30
n = 30
ys = [dp_count(n, x) for x in range(n + 1)]
print("nội suy khớp DP tại m=60:", lagrange_consecutive(ys, 60) == dp_count(n, 60))

đáp án: 122743172 | khớp cách 4: True | thời gian < 5s: True
nội suy khớp DP tại m=60: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nội suy với quá ít điểm (dùng `n` điểm thay vì `n + 1`)

Đa thức bậc `n` cần `n + 1` điểm. Thiếu một điểm thì nội suy ra đa thức bậc `n - 1` khác hẳn.

In [22]:
n, M = 4, 50
ys_wrong = [dp_count(n, x) for x in range(n)]          # SAI: chỉ n điểm
ys_right = [dp_count(n, x) for x in range(n + 1)]
print("sai:", lagrange_consecutive(ys_wrong, M), "| đúng:", lagrange_consecutive(ys_right, M), "| DP:", dp_count(n, M))

sai: 198450 | đúng: 2501450 | DP: 2501450


### Lỗi 2 — Quên đổi dấu `(-1)^{d-i}` trong mẫu số Lagrange

`Π_{j≠i}(i - j) = i! · (d - i)! · (-1)^{d-i}` — bỏ dấu thì kết quả sai hoàn toàn.

In [23]:
def lagrange_no_sign(ys, X, mod=MOD):
    d = len(ys) - 1
    res = 0
    for i in range(d + 1):
        num, den = 1, 1
        for j in range(d + 1):
            if j != i:
                num = num * (X - j) % mod
                den = den * abs(i - j) % mod          # SAI: bỏ mất dấu
        res = (res + ys[i] * num * pow(den, mod - 2, mod)) % mod
    return res

ys = [dp_count(3, x) for x in range(4)]
print("sai:", lagrange_no_sign(ys, 10), "| đúng:", lagrange_consecutive(ys, 10))

sai: 1830 | đúng: 570


### Lỗi 3 — Cho phép bước "bằng" hoặc quên tính luân phiên

Nếu DP dùng `Σ_{u ≤ v}` (cho phép bằng) hoặc cho bước lên nối tiếp bước lên, ta đếm cả mảng không ZigZag.

In [24]:
def dp_wrong_le(n, m):
    up = [v + 1 for v in range(m)]          # SAI: đếm cả u == v
    down = [m - v for v in range(m)]
    for _ in range(n - 2):
        pre = list(accumulate(down, initial=0))
        suf = list(accumulate(up, initial=0))
        up, down = [pre[v + 1] for v in range(m)], [suf[-1] - suf[v] for v in range(m)]
    return sum(up) + sum(down)

print("sai:", dp_wrong_le(3, 3), "| đúng:", dp_count(3, 3))

sai: 28 | đúng: 10


### Lỗi 4 — Lấy `M % MOD` rồi mới kiểm tra `M <= n`, hoặc tràn âm khi `(M - i)`

Trong Python không tràn số, nhưng nếu so sánh `M <= n` sau khi đã lấy modulo, hoặc quên `% MOD` khi `M - i` âm (không xảy ra ở đây vì `M > n`), sẽ ra kết quả sai. Luôn so sánh với `M` gốc, rồi mới chuyển sang modulo.

In [25]:
M = MOD + 2           # đoạn giả định cực lớn (vượt ràng buộc, chỉ để minh họa)
print("M % MOD =", M % MOD, "<= n?", M % MOD <= 3, "nhưng M thực sự lớn:", M > 3)

M % MOD = 2 <= n? True nhưng M thực sự lớn: True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Giá trị trong `[l, r]` khổng lồ, nhưng chỉ so sánh tương đối"** ⇒ đáp án là hàm của `m = r - l + 1`, và thường là **đa thức theo `m`** (bậc ≤ số phần tử). ⇒ Tính vài giá trị nhỏ bằng DP rồi **nội suy Lagrange**.
- Bộ ba bài cùng loạt cho thấy **mỗi ràng buộc gợi một kỹ thuật**:
  - `n, m` vừa phải ⇒ DP + tổng tiền tố;
  - `n` khổng lồ, `m` nhỏ ⇒ **lũy thừa ma trận**;
  - `m` khổng lồ, `n` nhỏ ⇒ **đa thức + nội suy**.
- Muốn kiểm chứng giả thuyết "là đa thức": tính sai phân bậc cao bằng code — nếu sai phân bậc `n + 1` bằng 0 thì đúng.

In [26]:
# Kiểm chứng tính đa thức bằng sai phân
def diffs(seq):
    return [b - a for a, b in zip(seq, seq[1:])]
n = 4
seq = [dp_count(n, x) for x in range(12)]
for k in range(n + 2):
    print(f"sai phân bậc {k}:", seq[:8])
    seq = diffs(seq)

sai phân bậc 0: [0, 0, 2, 16, 62, 170, 380, 742]
sai phân bậc 1: [0, 2, 14, 46, 108, 210, 362, 574]
sai phân bậc 2: [2, 12, 32, 62, 102, 152, 212, 282]
sai phân bậc 3: [10, 20, 30, 40, 50, 60, 70, 80]
sai phân bậc 4: [10, 10, 10, 10, 10, 10, 10, 10]
sai phân bậc 5: [0, 0, 0, 0, 0, 0, 0]


## 15. Biến thể của bài

1. **ZigZag I** (`n, m ≤ 2000`): Cách 2 là đủ.
2. **ZigZag II** (`n` tới `10^9`, `m` nhỏ): biểu diễn bước DP bằng ma trận `2m × 2m` rồi lũy thừa nhanh.
3. **Chỉ đếm dạng "lên–xuống"** (alternating permutation trên tập đa giá trị): bằng nửa đáp án (đối xứng).
4. **Hoán vị zigzag** (`m = n`, giá trị phân biệt): số Euler zigzag `E_n` (dãy *up/down numbers*), chính là `g_n(n)` ở Cách 4.
5. **Cho phép bằng nhau** hoặc **cấm thêm mẫu khác**: vẫn là đa thức theo `m`, nội suy vẫn dùng được.

In [27]:
# Biến thể 4: số hoán vị zigzag (giá trị phân biệt 1..n) = g_n(n) — so với số Euler zigzag 2*E_n
from math import comb
euler_up_down = [1, 1, 1, 2, 5, 16, 61, 272]      # E_0..E_7
for n in range(3, 8):
    f = [dp_count(n, x) for x in range(n + 1)]
    g_n = sum((-1) ** (n - i) * comb(n, i) * f[i] for i in range(n + 1))
    print(n, "g_n(n) =", g_n, "| 2*E_n =", 2 * euler_up_down[n])

3 g_n(n) = 4 | 2*E_n = 4
4 g_n(n) = 10 | 2*E_n = 10
5 g_n(n) = 32 | 2*E_n = 32
6 g_n(n) = 122 | 2*E_n = 122
7 g_n(n) = 544 | 2*E_n = 544


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Liệt kê `m^n` mảng | `O(m^n n)` | `O(n)` | Kiểm thử, tí hon |
| 2. DP + tiền tố | `up/down` theo giá trị cuối | `O(nm)` | `O(m)` | `m` nhỏ (ZigZag I) |
| 3. DP + Lagrange | Đa thức bậc ≤ n, nội suy | `O(n³)` | `O(n)` | **Lời giải chuẩn** khi `m` lớn |
| 4. Cơ sở nhị thức | `f(m) = Σ C(m,j) g(j)` | `O(n³)` | `O(n²)` | Khi cần ý nghĩa tổ hợp `g(j)` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `zigZagArrays(3, 1, 4)` = ? (gợi ý công thức `n = 3`).

**Bài 2 (tính tay).** Đếm tay số mảng ZigZag độ dài 4 với `m = 2`.

**Bài 3 (điền TODO).** Hoàn thiện `sum_powers(k, N)` = `1^k + 2^k + … + N^k mod p` bằng nội suy Lagrange (đa thức bậc `k + 1` theo `N`).

**Bài 4 (tự viết lại).** Viết lại `count(m)` chỉ dùng **một** mảng nhờ đối xứng `down[v] = up[m-1-v]`.

**Bài 5 (mini challenge).** Đếm mảng độ dài `n`, giá trị trong `[l, r]`, kề nhau **khác nhau** (không cần zigzag) bằng nội suy và kiểm chứng với công thức `m·(m-1)^{n-1}`.

In [28]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def sum_powers(k, N):
    # TODO: tính ys[x] = Σ_{i=1..x} i^k cho x = 0..k+1, rồi lagrange_consecutive(ys, N)
    pass

In [29]:
# Bài 4 — TODO (không gọi hàm này khi chạy notebook)
def count_one_array(n, m):
    # TODO: chỉ giữ up; down[v] = up[m-1-v]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `m = 4`. (2) `f3(m) = 2·(m-1)m(2m-1)/6`. (3) `= 2·3·4·7/6 = 28`.

**Bài 2.** (1) Chỉ có hai giá trị nên phải luân phiên `1,2,1,2` hoặc `2,1,2,1`. (2) Không có cách nào khác. (3) Đáp án `2`.

**Bài 3.** (1) `Σ i^k` là đa thức bậc `k + 1`. (2) Cần `k + 2` điểm `x = 0..k+1`. (3) Dùng `lagrange_consecutive`.

**Bài 4.** (1) Phản xạ `v ↦ m-1-v` đổi lên ↔ xuống. (2) `up_new[v] = Σ_{u<v} up[m-1-u] = Σ_{w > m-1-v} up[w]`. (3) Đáp án `2·Σ up`.

**Bài 5.** (1) DP: `cnt_n(m) = m(m-1)^{n-1}` là đa thức bậc `n`. (2) Tính `n+1` giá trị. (3) Nội suy và so với công thức.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1, 2
assert Solution().zigZagArrays(3, 1, 4) == 28 == brute(3, 1, 4)
assert brute(4, 1, 2) == 2 == Solution().zigZagArrays(4, 1, 2)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [31]:
# Bài 3
def sum_powers_sol(k, N):
    ys, s = [], 0
    for x in range(k + 2):
        if x > 0:
            s = (s + pow(x, k, MOD)) % MOD
        ys.append(s)
    return lagrange_consecutive(ys, N)

assert sum_powers_sol(1, 100) == 5050
assert sum_powers_sol(2, 10) == 385
assert sum_powers_sol(3, 1000) == sum(i**3 for i in range(1, 1001)) % MOD
print("Bài 3 OK")

Bài 3 OK


In [32]:
# Bài 4
def count_one_array_sol(n, m):
    if m <= 1:
        return 0
    up = list(range(m))
    for _ in range(n - 2):
        suf = list(accumulate(up, initial=0))      # suf[i] = Σ_{w<i} up[w]
        tot = suf[-1]
        up = [(tot - suf[m - v]) % MOD for v in range(m)]   # Σ_{w >= m-v} up[w] = Σ_{w > m-1-v}
    return 2 * sum(up) % MOD

for n in range(3, 9):
    for m in range(1, 9):
        assert count_one_array_sol(n, m) == dp_count(n, m)
print("Bài 4 OK")

Bài 4 OK


In [33]:
# Bài 5
def adjacent_distinct(n, l, r):
    M = r - l + 1
    def cnt(m):
        # DP: dp = số mảng độ dài i, mỗi bước chọn giá trị khác giá trị trước
        if m == 0:
            return 0
        total = m
        for _ in range(n - 1):
            total = total * (m - 1)
        return total % MOD
    ys = [cnt(x) for x in range(n + 1)]
    return lagrange_consecutive(ys, M)

for n in range(1, 8):
    for M in (5, 37, 10**9):
        assert adjacent_distinct(n, 1, M) == M * pow(M - 1, n - 1, MOD) % MOD
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# 1) DP nhỏ cho m = 0..n (đa thức bậc <= n theo m)
ys = [count(x) for x in range(n + 1)]
# 2) Nội suy Lagrange tại M với các điểm 0..d
def lagrange(ys, X):
    d = len(ys) - 1
    if X <= d: return ys[X]
    pre/suf = tích (X - i); fact = giai thừa
    term_i = ys[i] * pre[i] * suf[i+1] / (i! (d-i)!) * (-1)^(d-i)
```

**Checklist:**

- [ ] Đã chuẩn hóa về `m = r - l + 1`?
- [ ] DP chặn bước "bằng" và bảo đảm luân phiên lên/xuống?
- [ ] Đủ `n + 1` điểm nội suy (bậc ≤ `n`)?
- [ ] Đúng dấu `(-1)^{d-i}` trong Lagrange?
- [ ] So sánh `M <= n` với `M` gốc, không phải `M % MOD`?
- [ ] Mọi phép nhân đều `% MOD`; nghịch đảo bằng `pow(x, MOD-2, MOD)`?
- [ ] (Premium) Đã đối chiếu tên phương thức `zigZagArrays` với skeleton gốc?